# 05. Joining the three markets
The models use the Irish market **and** the US and German markets. In this notebook we put them in one table — carefully, so that no information from the future is used.

* **Needs:** `work/iseq_features.csv` and the clean foreign files. **Makes:** `work/dataset.csv`.

*Simple notebook series of the project 'This Time Is Different?'. Prepared with AI assistance (declared in `notes/ai_use.md`); a study notebook, not dissertation text.*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
folder = '/content/drive/MyDrive/DataSets/ThisTimeIsDifferent/'

import warnings
warnings.filterwarnings('ignore')     # hide warning messages

In [2]:
from pandas import read_csv
data1 = read_csv(folder + 'work/iseq_features.csv', index_col=0, parse_dates=True)
data2 = read_csv(folder + 'work/clean_GSPC.csv', index_col=0, parse_dates=True)
data3 = read_csv(folder + 'work/clean_GDAXI.csv', index_col=0, parse_dates=True)
data4 = read_csv(folder + 'work/clean_FTSE.csv', index_col=0, parse_dates=True)
print(data1.shape, data2.shape, data3.shape, data4.shape)

(5898, 10) (5850, 1) (5903, 1) (5873, 1)


## Why we must be careful: trading hours (Irish time)

| Market | Closes at |
|---|---|
| Dublin (ISEQ), London (FTSE), Frankfurt (DAX) | about 16:30 |
| New York (S&P 500) | about **21:00** |

The forecast for the next week is made at **08:00 the next morning**. By then the closes of **all** markets on day *t* are known, so we may use them. We may **never** use a value from day *t+1*.

## Returns of the foreign markets
The same two numbers as for the ISEQ: the return and the squared return.

In [3]:
from numpy import log
data2['us_ret'] = log(data2['Close']).diff()
data2['us_r2'] = data2['us_ret'] ** 2
data3['euro_ret'] = log(data3['Close']).diff()
data3['euro_r2'] = data3['euro_ret'] ** 2
data4['uk_ret'] = log(data4['Close']).diff()
data4['uk_r2'] = data4['uk_ret'] ** 2
data2.head()

,Close,us_ret,us_r2
2002-10-01,847.909973,NaN,NaN
2002-10-02,827.909973,-0.023870,0.000570
2002-10-03,818.950012,-0.010881,0.000118
2002-10-04,800.580017,-0.022687,0.000515
2002-10-07,785.280029,-0.019296,0.000372


## A small example of `merge_asof`
`merge_asof` joins two tables by date. For every Irish day it takes the **latest** US value dated **on or before** that day (`direction='backward'`). If New York was closed (4 July), the last known value is used again.

In [4]:
from pandas import DataFrame, to_datetime, merge_asof
irish = DataFrame({'Date': to_datetime(['2016-07-01', '2016-07-04', '2016-07-05']), 'irish_value': [1, 2, 3]})
us = DataFrame({'Date': to_datetime(['2016-07-01', '2016-07-05']), 'us_value': [10, 30]})   # 4 July: New York closed
merge_asof(irish, us, on='Date', direction='backward')

,Date,irish_value,us_value
0,2016-07-01,1,10
1,2016-07-04,2,10
2,2016-07-05,3,30


On 4 July the Irish row gets the US value of 1 July. Nothing from the future is used.

## Join the real data

In [5]:
dataset = merge_asof(data1, data2[['us_ret', 'us_r2']], left_index=True, right_index=True, direction='backward')
dataset = merge_asof(dataset, data3[['euro_ret', 'euro_r2']], left_index=True, right_index=True, direction='backward')
dataset = merge_asof(dataset, data4[['uk_ret', 'uk_r2']], left_index=True, right_index=True, direction='backward')
dataset.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 5898 entries, 2002-10-01 to 2025-12-30
Data columns (total 16 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   High         5898 non-null   float64
 1   Low          5898 non-null   float64
 2   Close        5898 non-null   float64
 3   ret          5897 non-null   float64
 4   r2           5897 non-null   float64
 5   rv5_fwd      5893 non-null   float64
 6   log_rv5_fwd  5893 non-null   float64
 7   rv_d         5897 non-null   float64
 8   rv_w         5893 non-null   float64
 9   rv_m         5876 non-null   float64
 10  us_ret       5897 non-null   float64
 11  us_r2        5897 non-null   float64
 12  euro_ret     5897 non-null   float64
 13  euro_r2      5897 non-null   float64
 14  uk_ret       5897 non-null   float64
 15  uk_r2        5897 non-null   float64
dtypes: float64(16)
memory usage: 783.3 KB


How often did we have to re-use an older foreign value? (`isin` checks if each Irish date is also a US date; `~` means *not*.)

In [6]:
print('US  :', round((~dataset.index.isin(data2.index)).mean() * 100, 1), '% of Irish days')
print('euro:', round((~dataset.index.isin(data3.index)).mean() * 100, 1), '% of Irish days')
print('UK  :', round((~dataset.index.isin(data4.index)).mean() * 100, 1), '% of Irish days')

US  : 2.6 % of Irish days
euro: 0.9 % of Irish days
UK  : 0.9 % of Irish days


## The sample starts in January 2003
The data begin in October 2002 only so that the 22-day averages are complete on the first day. The study itself starts after the dot-com crash, in **January 2003**.

In [7]:
dataset = dataset.loc['2003-01-01':]
dataset.head()

,High,Low,Close,ret,r2,rv5_fwd,log_rv5_fwd,rv_d,rv_w,rv_m,us_ret,us_r2,euro_ret,euro_r2,uk_ret,uk_r2
2003-01-02,4113.229980,3988.000000,4109.430176,0.028233,0.000797,0.000257,-8.265859,0.000797,0.000171,0.000167,0.032661,1.066727e-03,0.070861,0.005021,0.017384,3.022149e-04
2003-01-03,4186.310059,4109.560059,4165.959961,0.013662,0.000187,0.000132,-8.934309,0.000187,0.000207,0.000173,-0.000484,2.344034e-07,-0.003905,0.000015,-0.001123,1.261050e-06
2003-01-06,4195.560059,4117.419922,4188.959961,0.005506,0.000030,0.000146,-8.830180,0.000030,0.000211,0.000170,0.022226,4.939745e-04,0.020579,0.000424,-0.000899,8.087496e-07
2003-01-07,4199.669922,4174.149902,4183.220215,-0.001371,0.000002,0.000148,-8.818115,0.000002,0.000210,0.000170,-0.006566,4.311406e-05,-0.014188,0.000201,-0.011057,1.222585e-04
2003-01-08,4183.259766,4122.149902,4157.870117,-0.006078,0.000037,0.000374,-7.892551,0.000037,0.000211,0.000156,-0.014186,2.012346e-04,-0.039237,0.001540,-0.008272,6.842288e-05


## How strongly do the markets move together? (2003 – August 2023)

In [8]:
dataset.loc[:'2023-08-31', ['ret', 'us_ret', 'euro_ret', 'uk_ret']].corr().round(2)

,ret,us_ret,euro_ret,uk_ret
ret,1.00,0.48,0.69,0.71
us_ret,0.48,1.00,0.60,0.56
euro_ret,0.69,0.60,1.00,0.83
uk_ret,0.71,0.56,0.83,1.00


The Irish market moves most with London (0.71) and Frankfurt (0.69), less with New York (0.48). London and Frankfurt move together a lot (0.83) — **too much** to use both as inputs (SHAP could not tell them apart). So the main models use the **DAX**, and the FTSE is kept for a robustness check.

In [9]:
dataset.to_csv(folder + 'work/dataset.csv')
print(dataset.shape)

(5835, 16)
